# Task 08 — Error-aware modelling

## 🌌 Macrocosm photo-z — outlier study (tasks 2026-06-16)

**✅ GIVEN (do NOT re-derive this):** we already ran a **400k 5-fold cross-validation** with three
models (HGB, RF, MLP) and collected the galaxies that *all three* predict catastrophically out-of-fold
(|Δz/(1+z)| > 0.05). There are **6,974** such **hard** galaxies. Their objids are in
**`../hard_objids.csv`**. Treat this hard set as a **fixed input** — your job is to characterize / act
on it, not to re-find it from 400k.

**Catalog:** `gs://macrocosm-lewagon/data/sample_v1/catalog_v1.parquet` (600k rows, 55 columns).
The SDSS **`-9999` sentinel** means "not measured" — always clean it to NaN first.

**Metric:** `σ_MAD = 1.4826 · median(|Δz − median(Δz)|)`, with `Δz = (z_pred − z_true)/(1+z)`;
an **outlier** is `|Δz| > 0.05`. σ_MAD is the headline metric, report **outlier rate** alongside it.

---

Task 04 should show the hard set is dominated by photometric noise. Test whether telling the model about
its own uncertainty helps: add the per-band `modelMagErr_*` as features (or weight samples by 1/err²).

## ⚠️ Dependency

This task builds on **Task 02 — baseline models** and **Task 04 — hard-set-anatomy**. Before you start, make sure the prerequisite is **completed — by you or a teammate — and merged into `2026.6.16`**. If it isn't ready yet, coordinate with whoever owns it to finish the prerequisite first; or, of course, just wait until it lands.

In [2]:
# === shared setup: load catalog, clean -9999, build the 16 features ===
import pandas as pd, numpy as np

CATALOG = "/Users/mario/code/Macrocosm/notebooks/tasks-2026-6-16/catalog_v1.parquet"

FEATS = ["dered_u","dered_g","dered_r","dered_i","dered_z","g-r","u-g","r-i","i-z",
         "log_expRad_r","log_deVRad_r","log_petroRad_r","log_petroR50_r","log_petroR90_r",
         "fracDeV_r","conc_r"]

def load_features(path=CATALOG, n=None, seed=0):
    cat = pd.read_parquet(path)
    num = cat.select_dtypes("number").columns
    cat[num] = cat[num].mask(cat[num] <= -100)
    for a, b in [("u","g"),("g","r"),("r","i"),("i","z")]:
        cat[f"{a}-{b}"] = (cat[f"dered_{a}"] - cat[f"dered_{b}"]).clip(-1, 4)
    for s in ["expRad_r","deVRad_r","petroRad_r","petroR50_r","petroR90_r"]:
        cat["log_"+s] = np.log1p(cat[s].clip(lower=0))
    cat["conc_r"] = cat["petroR90_r"] / cat["petroR50_r"].replace(0, np.nan)
    D = cat[FEATS + ["redshift","objid"]].replace([np.inf,-np.inf], np.nan).dropna()
    if n:
        D = D.sample(n, random_state=seed).reset_index(drop=True)
    return D, cat

def smad(dz): return 1.4826 * np.median(np.abs(dz - np.median(dz)))

def metrics(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    dz = (y_pred - y_true) / (1 + y_true)
    return {"MAE": round(float(np.mean(np.abs(y_pred-y_true))), 5),
            "sigma_MAD": round(float(smad(dz)), 5),
            "outlier_rate": round(float(np.mean(np.abs(dz) > 0.05)), 5)}

HARD = set(pd.read_csv("../hard_objids.csv")["objid"])
print(f"hard set: {len(HARD)} galaxies")

hard set: 6974 galaxies


❓ **Question (add error features)** ❓

👇 Build the feature set, then ALSO append the 5 `modelMagErr_*` columns (cleaned). Train the best model from Task 02, 100k (`seed=1`), `KFold(3, ...,42)`.

In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold

ERR_FEATS = ["modelMagErr_u","modelMagErr_g","modelMagErr_r","modelMagErr_i","modelMagErr_z"]
FEATS_ERR = FEATS + ERR_FEATS

D, cat = load_features(n=100000, seed=1)

# Merge the 5 error columns onto D
err_cols = cat[["objid"] + ERR_FEATS].copy()
D_err = D.merge(err_cols, on="objid")
D_err[ERR_FEATS] = D_err[ERR_FEATS].replace([np.inf, -np.inf], np.nan)
D_err = D_err.dropna(subset=ERR_FEATS).reset_index(drop=True)
print(f"Dataset after dropping missing errors: {len(D_err):,} rows "
      f"(dropped {len(D)-len(D_err):,})")

kf = KFold(3, shuffle=True, random_state=42)
rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)

# Baseline: 16 features (same rows as D_err for fair comparison)
oof_base = np.full(len(D_err), np.nan)
for tr, val in kf.split(D_err):
    rf.fit(D_err.iloc[tr][FEATS], D_err.iloc[tr]["redshift"])
    oof_base[val] = rf.predict(D_err.iloc[val][FEATS])

# Error-aware: 16 + 5 error features
oof_err = np.full(len(D_err), np.nan)
for tr, val in kf.split(D_err):
    rf.fit(D_err.iloc[tr][FEATS_ERR], D_err.iloc[tr]["redshift"])
    oof_err[val] = rf.predict(D_err.iloc[val][FEATS_ERR])

print("\nBaseline    (16 feats):", metrics(D_err["redshift"], oof_base))
print("Error-aware (21 feats):", metrics(D_err["redshift"], oof_err))

Dataset after dropping missing errors: 100,000 rows (dropped 0)

Baseline    (16 feats): {'MAE': 0.01639, 'sigma_MAD': 0.01439, 'outlier_rate': 0.0309}
Error-aware (21 feats): {'MAE': 0.01636, 'sigma_MAD': 0.01435, 'outlier_rate': 0.03052}


❓ **Question (does it help)** ❓

👇 Compare σ_MAD and outlier rate against the Task 02 baseline (no error features). Split by normal vs hard if you like. Did error-awareness help the normal galaxies?

In [4]:
D_err["hard"] = D_err["objid"].isin(HARD)
D_err["oof_base"] = oof_base
D_err["oof_err"]  = oof_err

print(f"{'Group':<10} {'n':>7}  {'base σ_MAD':>10}  {'err σ_MAD':>10}  {'Δσ_MAD':>8}  "
      f"{'base out%':>9}  {'err out%':>8}  {'Δout%':>7}")
print("-" * 80)
for label, mask in [("ALL",    slice(None)),
                    ("normal", ~D_err["hard"]),
                    ("hard",   D_err["hard"])]:
    sub = D_err[mask] if label != "ALL" else D_err
    m_b = metrics(sub["redshift"], sub["oof_base"])
    m_e = metrics(sub["redshift"], sub["oof_err"])
    print(f"{label:<10} {len(sub):>7}  {m_b['sigma_MAD']:>10.5f}  {m_e['sigma_MAD']:>10.5f}  "
          f"{m_e['sigma_MAD']-m_b['sigma_MAD']:>+8.5f}  "
          f"{m_b['outlier_rate']:>9.4f}  {m_e['outlier_rate']:>8.4f}  "
          f"{m_e['outlier_rate']-m_b['outlier_rate']:>+7.4f}")

Group            n  base σ_MAD   err σ_MAD    Δσ_MAD  base out%  err out%    Δout%
--------------------------------------------------------------------------------
ALL         100000     0.01439     0.01435  -0.00004     0.0309    0.0305  -0.0004
normal       98856     0.01418     0.01413  -0.00005     0.0207    0.0206  -0.0001
hard          1144     0.10363     0.10891  +0.00528     0.9108    0.8899  -0.0210


## 📝 Write your report

In [5]:
# === write_report: run this after you've filled in your results, it generates report.md ===
def write_report(title, results: dict, conclusion: str, path="report.md"):
    lines = [f"# {title}", "", "_Auto-generated by task.ipynb — fill `results` and `conclusion` above._", "",
             "## Results", ""]
    for k, v in results.items():
        lines.append(f"- **{k}**: {v}")
    lines += ["", "## Conclusion", "", conclusion, ""]
    with open(path, "w") as f:
        f.write("\n".join(lines))
    print("wrote", path)

In [6]:
write_report("Task 08 — Error-aware modelling",
             {"baseline sigma_MAD (16 feats, all)":       "0.01439  |  outlier rate: 0.0309",
              "error-aware sigma_MAD (21 feats, all)":    "0.01435  |  outlier rate: 0.0305",
              "outlier rate change (all)":                "-0.0004 (−0.4 pp) — marginal",
              "normal galaxies sigma_MAD change":         "0.01418 → 0.01413  (Δ−0.00005)",
              "hard galaxies sigma_MAD change":           "0.10363 → 0.10891  (Δ+0.00528, worse)",
              "hard galaxies outlier rate change":        "0.9108 → 0.8899  (Δ−0.021, −2.1 pp)"},
             "Adding the 5 modelMagErr_* columns as features produces no meaningful improvement. "
             "Overall σ_MAD drops by only 0.00004 (0.01439 → 0.01435) and outlier rate by 0.04 pp. "
             "For normal (bright) galaxies the gain is negligible (Δσ_MAD = −0.00005). "
             "For hard (faint, noisy) galaxies the outlier rate falls slightly (91.1% → 89.0%, −2.1 pp), "
             "but σ_MAD actually worsens (+0.005), suggesting the model is not learning useful signal "
             "from the error columns — just fitting noise. "
             "Conclusion: photometric errors tell the model *that* a galaxy is hard, but not *where* to "
             "put its prediction. The hard set is intrinsically difficult: faint galaxies lack reliable "
             "color information and no amount of uncertainty flagging recovers it. "
             "Better strategies would be to exclude hard galaxies from training (Task 05) or use "
             "deeper/additional photometric bands."
             )

wrote report.md


In [ ]:
# === Commit & push your results (run last; make sure you are on branch 2026.6.16) ===
# First time: git pull origin 2026.6.16   to get the latest.
!git add task.ipynb report.md && git commit -m "08-error-aware-features task" && git push origin 2026.6.16

## 🔭 Go further (optional)

Play around with the data and the results you now have. If you find anything new — an unexpected pattern, a useful feature, a failure mode we missed — write it up and post it to our **YouTrack Knowledge Base** so the whole team benefits.